# Use Case 10: Horse Segmentation using U-Net

This notebook predicts a binary mask for every image pixel.

Each code cell is preceded by Markdown that explains what the step does, why it is needed, and which deep-learning concept is being demonstrated.


## Step 1: Import libraries

TensorFlow layers for the U-Net encoder, decoder and skip connections are imported.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Conv2DTranspose,
    Concatenate,
)


## Step 2: Locate image-mask pairs

Each input image has a corresponding mask with the same filename.

In [ ]:
IMAGE_DIR = Path("../datasets/07_horse_segmentation/images")
MASK_DIR = Path("../datasets/07_horse_segmentation/masks")

image_paths = sorted(IMAGE_DIR.glob("*.png"))
mask_paths = sorted(MASK_DIR.glob("*.png"))

print("Images:", len(image_paths))
print("Masks:", len(mask_paths))


## Step 3: Display one image and mask

The mask contains foreground and background pixel labels.

In [ ]:
sample_image = tf.keras.utils.load_img(
    image_paths[0],
    target_size=(128, 128),
)

sample_mask = tf.keras.utils.load_img(
    mask_paths[0],
    target_size=(128, 128),
    color_mode="grayscale",
)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(sample_image)
plt.title("Input Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(sample_mask, cmap="gray")
plt.title("Mask")
plt.axis("off")

plt.show()


## Step 4: Build a paired TensorFlow dataset

The image is normalized to 0–1. The mask is converted to binary values.

In [ ]:
def load_pair(image_path, mask_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (128, 128))
    image = tf.cast(image, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(
        mask,
        (128, 128),
        method="nearest",
    )
    mask = tf.cast(mask > 127, tf.float32)

    return image, mask


dataset = tf.data.Dataset.from_tensor_slices(
    (
        [str(path) for path in image_paths],
        [str(path) for path in mask_paths],
    )
)

dataset = dataset.map(
    load_pair,
    num_parallel_calls=tf.data.AUTOTUNE,
)

dataset = dataset.shuffle(80, seed=42)

train_ds = dataset.take(64).batch(8)
val_ds = dataset.skip(64).batch(8)


## Step 5: Build a small U-Net

The encoder reduces dimensions, the decoder restores them, and skip connections preserve spatial detail.

In [ ]:
def conv_block(inputs, filters):
    x = Conv2D(
        filters,
        3,
        activation="relu",
        padding="same",
    )(inputs)

    x = Conv2D(
        filters,
        3,
        activation="relu",
        padding="same",
    )(x)

    return x


inputs = Input(shape=(128, 128, 3))

c1 = conv_block(inputs, 16)
p1 = MaxPooling2D()(c1)

c2 = conv_block(p1, 32)
p2 = MaxPooling2D()(c2)

bridge = conv_block(p2, 64)

u1 = Conv2DTranspose(
    32,
    2,
    strides=2,
    padding="same",
)(bridge)

u1 = Concatenate()([u1, c2])
c3 = conv_block(u1, 32)

u2 = Conv2DTranspose(
    16,
    2,
    strides=2,
    padding="same",
)(c3)

u2 = Concatenate()([u2, c1])
c4 = conv_block(u2, 16)

outputs = Conv2D(
    1,
    1,
    activation="sigmoid",
)(c4)

model = tf.keras.Model(inputs, outputs)
model.summary()


## Step 6: Compile and train

Binary cross-entropy compares every predicted mask pixel with the correct mask pixel.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
)


## Step 7: Visualize a predicted mask

A probability threshold of 0.5 converts the output into a binary segmentation mask.

In [ ]:
images, true_masks = next(iter(val_ds))
predicted_masks = model.predict(images)

predicted_binary = (
    predicted_masks[0, :, :, 0] >= 0.5
)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(images[0])
plt.title("Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(true_masks[0, :, :, 0], cmap="gray")
plt.title("True Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(predicted_binary, cmap="gray")
plt.title("Predicted Mask")
plt.axis("off")

plt.show()
